# Pythia-410m: initialisation of the full experiments (t = 0 checkpoints)

Every method compared, so it takes much longer than GaugeMLA alone (`pythia410m_gaugemla.ipynb`). **GaugeMLA, our method, is the arm `mla_w64_ours_unrope_cost_kl`** (kits `p410m_klgc` at 50M and `p410m_kl5m` at 5M): each un-roped rotary plane is read out by its head's KL stand-in. `mla_w64_ours_unrope_cost` is **A_p^ramp, an ablation**: GaugeMLA without the KL stand-in, which un-ropes each plane by its average rotation over the offsets instead (the earlier version of the method). It is built and trained with the baselines, in `p410m_gc`.

Requires **Google Colab Pro+**. **Runtime: CPU, High-RAM** (Runtime → Change runtime type). No GPU is used.

The full experiments need **at least 90 GB free on Google Drive**: this notebook's 36 checkpoints take about 60 GB, and `pythia410m_train_50M.ipynb` saves 15 trained models, about 25 GB more.

Run the cells in order. GaugeMLA's cell fits F_i and Σ; every later cell copies those fits, so every method is built on the same ones, as in the paper:

1. **GaugeMLA**, 50M grid (`p410m_klgc`): fits F_i and Σ, measures the KL stand-in and builds its 3 checkpoints into `MyDrive/gaugemla/kl_standin_50m/p410m_klgc/`.
2. **The baselines and A_p^ramp**, 50M grid (`p410m_gc`): TransMLA, MHA2MLA, CARE and A_p^ramp, 12 checkpoints, into `MyDrive/gaugemla/grad_clip/p410m_gc/`.
3. **GaugeMLA**, 5M grid (`p410m_kl5m`): copies GaugeMLA's fits, KL stand-in and checkpoints into `MyDrive/gaugemla/kl_standin_5m/p410m_kl5m/`; it builds nothing.
4. **The other constructions**, 5M grid (`p410m_gc5m`): Decoupled, KL objective, GQA (pool), GQA + optimal gauge, Procrustes alignment, Per-group subspace, Joint key, joint value subspace, into `MyDrive/gaugemla/grad_clip_5m/p410m_gc5m/`.

Finished checkpoints are skipped, so a rerun continues where it stopped.

Requires a Hugging Face token: add it as the Colab secret `HF_TOKEN` and give this notebook access.

In [ ]:
# @title Setup
DRY_RUN = False  # @param {type:"boolean"}

import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
# The repository is public. While it is private, uncomment these lines and add the Colab secret READ_MLA_TOKEN
# (a GitHub token that can read it):
# from google.colab import userdata
# os.environ["READ_MLA_TOKEN"] = userdata.get("READ_MLA_TOKEN")
# with open("/content/askpass.sh", "w") as f:
#     f.write('#!/bin/sh\necho "$READ_MLA_TOKEN"\n')
# os.chmod("/content/askpass.sh", 0o700)
# os.environ["GIT_ASKPASS"] = "/content/askpass.sh"
os.environ["GIT_TERMINAL_PROMPT"] = "0"
if os.path.isdir("/content/repo/.git"):
    subprocess.run(["git", "-C", "/content/repo", "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", "/content/repo", "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                    "https://github.com/anon5159753/gaugemla.git", "/content/repo"], check=True)
    subprocess.run(["git", "-C", "/content/repo", "sparse-checkout", "set", "src", "colab"], check=True)
sys.path.insert(0, "/content/repo/colab")
import launch

In [ ]:
# @title 1. GaugeMLA (our method), t = 0 checkpoints of the 50M grid (p410m_klgc)
launch.run("p410m_klgc", "init", DRY_RUN=DRY_RUN)

In [ ]:
# @title 2. The baselines and A_p^ramp (ablation), t = 0 checkpoints of the 50M grid (p410m_gc)
launch.run("p410m_gc", "init", DRY_RUN=DRY_RUN)

In [ ]:
# @title 3. GaugeMLA (our method), t = 0 checkpoints of the 5M grid (p410m_kl5m)
launch.run("p410m_kl5m", "init", DRY_RUN=DRY_RUN)

In [ ]:
# @title 4. The other constructions, t = 0 checkpoints of the 5M grid (p410m_gc5m)
launch.run("p410m_gc5m", "init", DRY_RUN=DRY_RUN, INIT_ARMS="decoupled_optwide_fisher_sigma_dkr64,pool,gauge3B,procrustes_simk,optimal,optwide")